Notebook is used to work with Deployment scripts using PowerShell and Azure CLI when needed.  If you would like to use Cloud Shell in Notebook see ref:  [Title](https://github.com/dotnet/interactive/blob/main/samples/notebooks/powershell/Docs/Interact-With-Azure-Cloud-Shell.ipynb)

In [21]:
#Login to Azure
#Use -Environment Parm to if you need to use Azure Gov.  AzureUSGovernmnet

Connect-AzAccount


To override which subscription Connect-AzAccount selects by default, use `Update-AzConfig -DefaultSubscriptionForLogin 00000000-0000-0000-0000-000000000000`. Go to https://go.microsoft.com/fwlink/?linkid=2200610 for more information.

Account                SubscriptionName TenantId                             Environment
-------                ---------------- --------                             -----------
frgarofa@microsoft.com DSE EDog         72f988bf-86f1-41af-91ab-2d7cd011db47 AzureCloud



In [22]:
#Select Subscription

#$subscription = "Azure FFL Internal Subscription FRGAROFA (GOV)"

$subscription = "Azure FFL Internal Subscription FRGAROFA"

Select-AzSubscription -Subscription $subscription



Name                                     Account        SubscriptionNa Environment    TenantId
                                                        me
----                                     -------        -------------- -----------    --------
Azure FFL Internal Subscription FRGAROF… frgarofa@micr… Azure FFL Int… AzureCloud     72f988bf-86f…



In [3]:
#Create Resource Group if not setup for data maangment zone:

$ResourceGroup = 'dmz-cloudscale-dev'

New-AzResourceGroup -Name $ResourceGroup -Location 'East US'


Confirm
Provided resource group already exists. Are you sure you want to update it?
[Y] Yes  [N] No  [S] Suspend  [?] Help(default is 'Y')

Error: Command cancelled.

In [3]:
#Check Resource Group 

$ResourceGroup = 'dmz-cloudscale-dev'

Get-AzResource -ResourceGroupName $ResourceGroup | Format-Table -AutoSize


Name                                                                               ResourceGroupNam
                                                                                   e
----                                                                               ----------------
purviewacct-5h7tsge77gt7i-dev                                                      dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev.nic.880b6bb7-4341-4ff2-a6e9-1e3f7b32456e dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev.nic.6fc3c8f6-477e-4cc2-98db-e15f85500ef6  dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-account-pep-dev                                          dmz-cloudscale-…
purviewacct-5h7tsge77gt7i-portal-pep-dev                                           dmz-cloudscale-…



In [8]:
#Set Param values for ARM deployment template

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\arm\DMZ\Purview\Purview.template.json'
    TemplateParameterFile = '..\arm\DMZ\Purview\Purview.parameters.json'
    Verbose = $true
}

In [9]:
#Test ARM deplkoyment
Test-AzResourceGroupDeployment @params



SubscriptionNotFound - The subscription '02cd7474-626e-4aa4-b21f-4856bdb92779' could not be found.



In [ ]:
#Deploy ARM Template

New-AzResourceGroupDeployment @params

In [ ]:
#Get list of Supported API versions for resource ProviderNamespace

Get-AzResourceProvider -ProviderNamespace Microsoft.Purview | Select-Object ProviderNamespace -ExpandProperty ResourceTypes | Select-Object * -ExpandProperty ApiVersions | Sort-Object -Descending

In [103]:
#Deploy Private DNS Zones

$params = @{
    ResourceGroupName = 'demo-core-vnet'
    TemplateFile = '..\bicep\DMZ\services\Network\privatednszone.bicep'
}

#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params -Verbose



VERBOSE: Using Bicep v0.16.1
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Network\privatednszone.bicep(15,7) : Warning no-unused-params: Parameter "tags" is declared but never used. [https://aka.ms/bicep/linter/no-unused-params]
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Network\privatednszone.bicep(37,5) : Warning no-unused-vars: Variable "vnetName" is declared but never used. [https://aka.ms/bicep/linter/no-unused-vars]
S:\Repos\GitHub\csa-inabox\deploy\bicep\DMZ\services\Network\privatednszone.bicep(47,51) : Warning prefer-interpolation: Use string interpolation instead of the concat function. [https://aka.ms/bicep/linter/prefer-interpolation]
VERBOSE: Performing the operation "Creating Deployment" on target "demo-core-vnet".
VERBOSE: 4:45:26 PM - Template is valid.
VERBOSE: 4:45:27 PM - Create template deployment 'privatednszone'
VERBOSE: 4:45:27 PM - Checking deployment status in 5 seconds
VERBOSE: 4:45:32 PM - Checking deployment status in 18 seconds
VERBOSE: 4:

In [ ]:
#Test and Deploy with Bicep

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\bicep\DMZ\services\Purview\purview.bicep'
    Verbose = $true
}
#Test-AzResourceGroupDeployment @params

New-AzResourceGroupDeployment @params

.\userscript.ps1 -zones @(privatelink.blob.core.windows.net,privatelink.dfs.core.windows.net,privatelink.file.core.windows.net,privatelink.queue.core.windows.net,privatelink.tab)


In [ ]:
$zones = [string[]]('privatelink.purview.azure.com','privatelink.purviewstudio.azure.com')
$rg = 'demo-core-vnet'


function GetExistingVNetLinks {
param([string[]] $zones, [string] $ResourceGroupName)
 foreach ($zone in $zones) {
            $existingLinks += Get-AzPrivateDnsVirtualNetworkLink -ZoneName $zone -ResourceGroupName $ResourceGroupName | Select Name, ZoneName, ResourceGroupName, VirtualNetworkId, VirtualNetworkLinkState
    }
   $existingLinks | ConvertTo-Json
}

GetExistingVNetLinks -zones $zones -ResourceGroupName $rg



In [81]:
..\bicep\DMZ\services\Network\scripts\userscript.ps1 -zones ['privatelink.purview.azure.com','privatelink.purviewstudio.azure.com','privatelink.queue.core.windows.net','privatelink.table.core.windows.net','privatelink.blob.core.windows.net','privatelink.file.core.windows.net'] -ResourceGroupName demo-core-vnet 

[
  {
    "Name": "corevnet",
    "ZoneName": "privatelink.purview.azure.com",
    "ResourceGroupName": "demo-core-vnet",
    "VirtualNetworkId": "/subscriptions/7d207b0b-97d1-4855-8e5c-f48f8315d4f5/resourceGroups/demo-core-vnet/providers/Microsoft.Network/virtualNetworks/demo-core-vnet",
    "VirtualNetworkLinkState": "Completed"
  },
  {
    "Name": "eastcorevnet",
    "ZoneName": "privatelink.purview.azure.com",
    "ResourceGroupName": "demo-core-vnet",
    "VirtualNetworkId": "/subscriptions/7d207b0b-97d1-4855-8e5c-f48f8315d4f5/resourceGroups/demo-core-vnet/providers/Microsoft.Network/virtualNetworks/demo-core-vnet-east",
    "VirtualNetworkLinkState": "Completed"
  },
  {
    "Name": "jl4xrsitk6spe",
    "ZoneName": "privatelink.purview.azure.com",
    "ResourceGroupName": "demo-core-vnet",
    "VirtualNetworkId": "/subscriptions/7d207b0b-97d1-4855-8e5c-f48f8315d4f5/resourceGroups/demo-core-vnet/providers/Microsoft.Network/virtualNetworks/demo-privatelink-vnet",
    "VirtualNetwo